# Drishti AI: train and benchmark the pose anomaly model on ShanghaiTech

Run this on **Colab with a GPU** (Runtime → Change runtime type → T4 GPU), then **Runtime → Run all**. Takes about 15–25 minutes.

What it does:
1. Downloads the ShanghaiTech Campus pose data and ground truth released with STG-NF (ICCV 2023).
2. Trains two temporal models, an LSTM autoencoder and a Transformer autoencoder, on normal training clips only.
3. Scores every test frame with the standard protocol and reports frame-level AUC, against a rule-based speed baseline and the published STG-NF result.
4. Saves both checkpoints plus `pose_ae_shanghaitech.pt` (the one the app loads, Transformer by default), `results.json`, and `roc_curve.png`, then downloads them.

In [ ]:
# 1. Get the code
!rm -rf /content/Drishti-AI && git clone -q https://github.com/theecstaticone7-cyber/Drishti-AI.git
!git clone -q https://github.com/orhir/STG-NF.git
!pip -q install gdown
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to GPU")

In [ ]:
# 2. Download ShanghaiTech poses + ground truth (STG-NF data release, ~1 GB)
import os, glob, shutil, zipfile
if not glob.glob("/content/STG-NF/data/ShanghaiTech/pose/train/*.json"):
    !gdown -q --fuzzy "https://drive.google.com/file/d/1o9h3Kh6zovW4FIHpNBGnYIRSbGCu-qPt/view" -O /content/stgnf_data.zip
    with zipfile.ZipFile("/content/stgnf_data.zip") as z:
        z.extractall("/content/stgnf_data")
    # find the folder that contains ShanghaiTech/pose/train, wherever the zip put it
    hits = glob.glob("/content/stgnf_data/**/ShanghaiTech/pose/train", recursive=True)
    assert hits, "Could not find ShanghaiTech/pose/train in the download"
    src = os.path.dirname(os.path.dirname(hits[0]))        # .../ShanghaiTech
    dst = "/content/STG-NF/data/ShanghaiTech"
    for sub in ("pose", "gt"):
        if os.path.exists(os.path.join(src, sub)):
            if os.path.exists(os.path.join(dst, sub)):
                shutil.rmtree(os.path.join(dst, sub))
            shutil.copytree(os.path.join(src, sub), os.path.join(dst, sub))
print("train clips:", len(glob.glob("/content/STG-NF/data/ShanghaiTech/pose/train/*.json")))
print("test clips: ", len(glob.glob("/content/STG-NF/data/ShanghaiTech/pose/test/*.json")))
print("gt masks:   ", len(glob.glob("/content/STG-NF/data/ShanghaiTech/gt/test_frame_mask/*.npy")))

If the Google Drive download fails with a quota error, open the link in your browser, download the zip,
upload it to Colab as `/content/stgnf_data.zip`, and run the cell again.

In [ ]:
# 3. Train + evaluate (writes into Drishti-AI/inference/models/)
!python -W ignore /content/Drishti-AI/training/train_eval_shanghaitech.py \
    --stgnf_dir /content/STG-NF --out_dir /content/Drishti-AI/inference/models --epochs 30

In [ ]:
# 4. Optional: reproduce the published STG-NF number with their checkpoint, for a same-machine comparison
!cd /content/STG-NF && sed -i "s/np\.int)/int)/; s/np\.int(/int(/" dataset.py utils/*.py && \
  sed -i "s/plt.style.use('seaborn-ticks')/pass/" utils/pose_utils.py && \
  python -W ignore train_eval.py --dataset ShanghaiTech --checkpoint checkpoints/ShanghaiTech_85_9.tar \
    --flow_permutation invcov 2>&1 | grep -i -E "auc|error" | tail -5

In [ ]:
# 5. Results
import json
from IPython.display import Image, display
r = json.load(open("/content/Drishti-AI/inference/models/results.json"))
print(f"{'Method':55s} {'AUC':>6s} {'Prec':>6s} {'Rec':>6s} {'F1':>6s} {'FalseAlarm':>10s}")
print(f"{'STG-NF, ICCV 2023 (published)':55s} {r['reference_published']['STG-NF (ICCV 2023), pose-only']*100:6.1f}")
for name, m in r["methods"].items():
    row = f"{name:55s} {m['frame_auc']*100:6.1f}"
    if "precision_at_threshold" in m:
        row += (f" {m['precision_at_threshold']:6.2f} {m['recall_at_threshold']:6.2f}"
                f" {m['f1_at_threshold']:6.2f} {m['false_alarm_rate_on_normal_frames']*100:9.1f}%")
    print(row)
print("\nApp uses:", r["deployed_model"])
display(Image("/content/Drishti-AI/inference/models/roc_curve.png"))


In [ ]:
# 6. Download the outputs, then put them in Drishti-AI/inference/models/ on your laptop
from google.colab import files
for f in ("pose_ae_shanghaitech.pt", "pose_ae_lstm.pt", "pose_ae_transformer.pt", "results.json", "roc_curve.png"):
    files.download(f"/content/Drishti-AI/inference/models/{f}")
